In [29]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,MultiLabelBinarizer
df = pd.DataFrame({
    'book_id':   [101, 102, 103, 104, 105, 106, 107, 108, 109],
    'format':    ['Paperback', 'Ebook', 'Hardcover', 'Ebook', 'Paperback',
                  'Audiobook', 'Hardcover', 'Ebook', 'Paperback'],
    'condition': ['Poor', 'Good', 'New', 'Fair', 'Good', 'New', 'Fair', 'Poor', 'New'],
    'language':  ['Tamil', 'English', 'Hindi', 'English', 'Tamil',
                  'English', 'Hindi', 'Tamil', 'English'],
    'tags':      [['Fiction'], ['Fiction', 'Thriller'], ['History'],
                  ['Thriller', 'Romance'], ['Fiction', 'History', 'Romance'],
                  ['Selfhelp'], ['History', 'Fiction'], ['Romance'],
                  ['Thriller', 'Selfhelp']],
    'in_stock':  ['Yes', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No']
})

In [30]:
#1. Label encode in_stock and print the class mapping.
lbl = LabelEncoder()
df['in_stock'] = lbl.fit_transform(df['in_stock'])
class_mapping = dict(zip(lbl.classes_, range(len(lbl.classes_))))
print("Class Mapping:", class_mapping)

Class Mapping: {'No': 0, 'Yes': 1}


In [31]:
#2. Ordinal encode condition with the order Poor < Fair < Good < New. Use .map() this time, not a list comprehension.
unique_condition = 'Poor < Fair < Good < New'.split(' < ')
conditions_mapping = {k:i for i,k in enumerate(unique_condition)}
df['condition'] = df['condition'].map(conditions_mapping)
df

,book_id,format,condition,language,tags,in_stock
0,101,Paperback,0,Tamil,[Fiction],1
1,102,Ebook,2,English,"[Fiction, Thriller]",0
2,103,Hardcover,3,Hindi,[History],1
3,104,Ebook,1,English,"[Thriller, Romance]",1
4,105,Paperback,2,Tamil,"[Fiction, History, Romance]",0
5,106,Audiobook,3,English,[Selfhelp],0
6,107,Hardcover,1,Hindi,"[History, Fiction]",1
7,108,Ebook,0,Tamil,[Romance],1
8,109,Paperback,3,English,"[Thriller, Selfhelp]",0


In [32]:
#3. One-hot format and language. Produce both versions, with and without drop_first, and keep the drop_first one. 
#Drop the original string columns after concat
format_language_one_hot_without_drop_first = pd.get_dummies(df[['format','language']],columns=['format','language'],dtype=int,prefix='withoutdrop',prefix_sep='__')
format_language_one_hot_with_drop_first = pd.get_dummies(df[['format','language']],columns=['format','language'],dtype=int,prefix='withdrop',drop_first=True,prefix_sep='__')
df=pd.concat([df,format_language_one_hot_without_drop_first,format_language_one_hot_with_drop_first],axis=1).drop(columns={'format','language'})
df

,book_id,condition,tags,in_stock,withoutdrop__Audiobook,withoutdrop__Ebook,withoutdrop__Hardcover,withoutdrop__Paperback,withoutdrop__English,withoutdrop__Hindi,withoutdrop__Tamil,withdrop__Ebook,withdrop__Hardcover,withdrop__Paperback,withdrop__Hindi,withdrop__Tamil
0,101,0,[Fiction],1,0,0,0,1,0,0,1,0,0,1,0,1
1,102,2,"[Fiction, Thriller]",0,0,1,0,0,1,0,0,1,0,0,0,0
2,103,3,[History],1,0,0,1,0,0,1,0,0,1,0,1,0
3,104,1,"[Thriller, Romance]",1,0,1,0,0,1,0,0,1,0,0,0,0
4,105,2,"[Fiction, History, Romance]",0,0,0,0,1,0,0,1,0,0,1,0,1
5,106,3,[Selfhelp],0,1,0,0,0,1,0,0,0,0,0,0,0
6,107,1,"[History, Fiction]",1,0,0,1,0,0,1,0,0,1,0,1,0
7,108,0,[Romance],1,0,1,0,0,0,0,1,1,0,0,0,1
8,109,3,"[Thriller, Selfhelp]",0,0,0,0,1,1,0,0,0,0,1,0,0


In [33]:
#4. Binarise tags with MultiLabelBinarizer. Print mlb.classes_ before joining.
mlb = MultiLabelBinarizer()
tags_encoded = mlb.fit_transform(df['tags'])
df_tags_encoded = pd.DataFrame(tags_encoded, columns=mlb.classes_, index=df.index)
df = pd.concat([df,df_tags_encoded],axis=1).drop(columns={'tags'})
df

,book_id,condition,in_stock,withoutdrop__Audiobook,withoutdrop__Ebook,withoutdrop__Hardcover,withoutdrop__Paperback,withoutdrop__English,withoutdrop__Hindi,withoutdrop__Tamil,withdrop__Ebook,withdrop__Hardcover,withdrop__Paperback,withdrop__Hindi,withdrop__Tamil,Fiction,History,Romance,Selfhelp,Thriller
0,101,0,1,0,0,0,1,0,0,1,0,0,1,0,1,1,0,0,0,0
1,102,2,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1
2,103,3,1,0,0,1,0,0,1,0,0,1,0,1,0,0,1,0,0,0
3,104,1,1,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1
4,105,2,0,0,0,0,1,0,0,1,0,0,1,0,1,1,1,1,0,0
5,106,3,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
6,107,1,1,0,0,1,0,0,1,0,0,1,0,1,0,1,1,0,0,0
7,108,0,1,0,1,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0
8,109,3,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,1,1


In [34]:
#5. Build final X and y. Drop book_id. Assert X has no non-numeric columns and that X and y have matching row counts.
df.drop(columns={'book_id'},inplace=True)
df

,condition,in_stock,withoutdrop__Audiobook,withoutdrop__Ebook,withoutdrop__Hardcover,withoutdrop__Paperback,withoutdrop__English,withoutdrop__Hindi,withoutdrop__Tamil,withdrop__Ebook,withdrop__Hardcover,withdrop__Paperback,withdrop__Hindi,withdrop__Tamil,Fiction,History,Romance,Selfhelp,Thriller
0,0,1,0,0,0,1,0,0,1,0,0,1,0,1,1,0,0,0,0
1,2,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1
2,3,1,0,0,1,0,0,1,0,0,1,0,1,0,0,1,0,0,0
3,1,1,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1
4,2,0,0,0,0,1,0,0,1,0,0,1,0,1,1,1,1,0,0
5,3,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
6,1,1,0,0,1,0,0,1,0,0,1,0,1,0,1,1,0,0,0
7,0,1,0,1,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0
8,3,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,1,1


In [35]:
import pandas as pd

df = pd.DataFrame({
    'plan':      ['Basic', 'Pro', 'Enterprise', 'Basic', 'Pro',
                  'Enterprise', 'Basic', 'Pro'],
    'tier':      ['Bronze', 'Gold', 'Platinum', 'Silver', 'Gold',
                  'Platinum', 'Bronze', 'Silver'],
    'region':    ['South', 'North', 'South', 'East', 'North',
                  'West', 'East', 'South'],
    'monthly_fee': [299, 999, 4999, 299, 999, 4999, 299, 999],
    'seats':       [2, 15, 120, 3, 20, 200, 1, 18],
    'support_hrs': [5, 40, 500, 8, 45, 800, 4, 38]
})

### Encoding

In [36]:
#Ordinal encode tier with order Bronze < Silver < Gold < Platinum, using .map().
unique_tier = 'Bronze < Silver < Gold < Platinum'.split(' < ')
tier_mapping = {k:i for i,k in enumerate(unique_tier)}
df['tier'] = df['tier'].map(tier_mapping)
df

,plan,tier,region,monthly_fee,seats,support_hrs
0,Basic,0,South,299,2,5
1,Pro,2,North,999,15,40
2,Enterprise,3,South,4999,120,500
3,Basic,1,East,299,3,8
4,Pro,2,North,999,20,45
5,Enterprise,3,West,4999,200,800
6,Basic,0,East,299,1,4
7,Pro,1,South,999,18,38
